In [1]:
import numpy as np
import tensorflow as tf
from scipy.signal import resample
import scipy.io as sio
import os

# Load the original .mat file
mat_path = '/Users/mczhang/Documents/GitHub/FM5_ML/02-data/D_man/D_CC1_CCPo.mat'
mat = sio.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
Felix = mat['Felix']  # 1D array of structs



2025-09-11 09:46:40.748438: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:

# Collect and preprocess waveforms
X, Y, indices = [], [], []

for idx, entry in enumerate(Felix):
    if not hasattr(entry, 'W1_CC1') or not hasattr(entry, 'CC1_CCPo'):
        continue
    waveform = entry.W1_CC1
    if waveform is None or len(waveform) != 128:
        continue
    waveform_resampled = resample(waveform, 600)
    polarity = np.sign(entry.CC1_CCPo)
    # if polarity == 0:
    #     continue
    X.append(waveform_resampled.astype(np.float32))
    Y.append(polarity)
    indices.append(idx)

X = np.stack(X)
Y = np.array(Y, dtype=np.int8)


In [3]:

# Preprocessing: demean and normalize
def demean(data, semiamp=200):
    lung = data.shape[1]
    return data - np.mean(data[:, lung//2 - semiamp : lung//2 - 5], axis=1, keepdims=True)

def normalize(data, threshold=20):
    lung = data.shape[1]
    sism_0_arr = data[:, 0:lung//2 - 5]
    norm_factor = threshold * np.maximum(np.max(sism_0_arr, axis=1), -np.min(sism_0_arr, axis=1)).reshape(-1, 1)
    data = np.clip(data / norm_factor, -1.0, 1.0)
    full_norm = np.maximum(np.max(data, axis=1), -np.min(data, axis=1)).reshape(-1, 1)
    return data / full_norm

X = demean(X)
X = normalize(X)

# Load model and predict
model = tf.keras.models.load_model('./Network_CFM/CFM_with_timeshift.hdf5')
semi_amp = X.shape[1] // 2
X_window = X[:, semi_amp - 80 : semi_amp + 80]
y_pred = model.predict(X_window, batch_size=16)

# Convert predicted probability to polarity: +1 = Up, -1 = Down
y_polarity = np.where(y_pred.flatten() >= 0.5, 1, -1)

# Save prediction into Felix struct under a new field `CC1_CFM_Po`
for i, idx in enumerate(indices):
    setattr(Felix[idx], 'CC1_CFM_Po', y_polarity[i])

# Save back to .mat
output_path = 'Felix_with_predictions_timeshift.mat'
sio.savemat(output_path, {'Felix': Felix})
print(f"Saved predictions in '{output_path}' with field 'CC1_CFM_Po'")

63/63 [==============================] - 0s 4ms/step
Saved predictions in 'Felix_with_predictions_timeshift.mat' with field 'CC1_CFM_Po'
